# 01 — Dataset comparison: `unsorted/` vs `medical_records/`

First step of EDA. Both directories under `data/` carry the same digital-stethoscope corpus, but in two different layouts and with different file coverage. This notebook quantifies that and surfaces the things we'll need to decide before training.

**Layout on disk (after extraction)**

| | `unsorted/` | `medical_records/` |
|---|---|---|
| Layout | 10 subdirectories (one per batch), each containing WAVs + a per-batch `merged-*.json` | 3009 loose WAVs in `ALL/` + one global `merged-1761225412230.json` |

`unsorted/` was originally extracted from `data/archives/all.tar` (kept as backup) into 10 zip files, then unzipped. Two of those zips (`1604_final.zip`, `3004_final.zip`) had no inner directory — their files spilled into the top level — and tar created 11 macOS metadata sidecars (`._*.zip`, ~478 B each). After cleanup: stray files were moved into `1604_final/` and `3004_final/`, sidecars deleted. Final: **10 clean subdirectories, 1640 wavs + 10 `merged*.json` files, no top-level junk.**

Both sources use the same JSON schema as the upstream `dnd` project: `{"files": [{uid, doctor, filename, markup: [{start, length, phase, quality, noise_level, breathe, pathology}]}]}` — the format the dnd training pipelines (`script/training/`, `1/`) consume directly.

In [ ]:
import os
import json
import wave
import contextlib
import random
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import numpy as np
import torchaudio
import torch
from IPython import display


DATA_ROOT = Path('../data')
UNSORTED  = DATA_ROOT / 'unsorted'
MR_DIR    = DATA_ROOT / 'medical_records'
MR_WAVS   = MR_DIR / 'ALL'
MR_JSON   = MR_DIR / 'merged-1761225412230.json'

PATHOLOGY_CLASSES = [
    'none', 'hard breathing', 'dry buzzing wheezing',
    'wet wheezing', 'dry whistling wheezing', 'crepitus',
]

pd.set_option('display.max_rows', 60)

## 1. Inventory

Walk each batch subdirectory under `unsorted/`, count its WAVs, and load its `merged-*.json` label file. Then do the same for `medical_records/`.

In [ ]:
def load_unsorted_batches():
    """Yield (batch_name, batch_size_bytes, wav_filenames, json_files_list) for each subdir."""
    for entry in sorted(os.listdir(UNSORTED)):
        sub = UNSORTED / entry
        if not sub.is_dir():
            continue
        wavs   = sorted(p.name for p in sub.iterdir() if p.suffix.lower() == '.wav')
        jsons  = sorted(p      for p in sub.iterdir() if p.suffix.lower() == '.json')
        size   = sum(p.stat().st_size for p in sub.iterdir() if p.is_file())
        files  = []
        if jsons:
            with open(jsons[0], encoding='utf-8-sig') as jf:
                files = json.load(jf).get('files', [])
        yield entry, size, wavs, files

rows = []
all_unsorted_files = []
for batch, size, wavs, files in load_unsorted_batches():
    rows.append({'batch': batch, 'size_MB': round(size / 1e6, 1),
                 'wavs_on_disk': len(wavs), 'json_entries': len(files)})
    all_unsorted_files.extend(files)

unsorted_inv = pd.DataFrame(rows)
unsorted_inv.loc['TOTAL'] = ['—', unsorted_inv['size_MB'].sum(),
                             unsorted_inv['wavs_on_disk'].sum(), unsorted_inv['json_entries'].sum()]
unsorted_inv

In [ ]:
mr_wav_files = sorted(f for f in os.listdir(MR_WAVS) if f.lower().endswith('.wav'))
mr_wav_bytes = sum((MR_WAVS / w).stat().st_size for w in mr_wav_files)
with open(MR_JSON, encoding='utf-8-sig') as f:
    mr_json = json.load(f)
mr_files = mr_json['files']

pd.DataFrame([
    {'side': 'unsorted/ (10 batches)',
     'on_disk_size_GB': round(unsorted_inv.loc['TOTAL', 'size_MB'] / 1000, 2),
     'wavs': unsorted_inv.loc['TOTAL', 'wavs_on_disk'],
     'json_entries': unsorted_inv.loc['TOTAL', 'json_entries']},
    {'side': 'medical_records/',
     'on_disk_size_GB': round(mr_wav_bytes / 1e9, 2),
     'wavs': len(mr_wav_files),
     'json_entries': len(mr_files)},
])

## 2. Overlap between the two sources

Are they containments? Disjoint? Partial overlap?

In [ ]:
unsorted_filenames = {e['filename'] for e in all_unsorted_files}
unsorted_uids      = {e['uid']      for e in all_unsorted_files}
mr_filenames       = {e['filename'] for e in mr_files}
mr_uids            = {e['uid']      for e in mr_files}
mr_wav_set         = set(mr_wav_files)

# Sanity: do unsorted JSONs match the wavs actually on disk?
unsorted_wavs_on_disk = set()
for sub in UNSORTED.iterdir():
    if sub.is_dir():
        unsorted_wavs_on_disk.update(p.name for p in sub.iterdir() if p.suffix.lower() == '.wav')

overlap = pd.DataFrame([
    {'metric': 'unique filenames in unsorted jsons',     'value': len(unsorted_filenames)},
    {'metric': 'unique wavs on disk in unsorted/',       'value': len(unsorted_wavs_on_disk)},
    {'metric': 'unsorted json filenames not on disk',    'value': len(unsorted_filenames - unsorted_wavs_on_disk)},
    {'metric': 'unsorted wavs on disk not in any json',  'value': len(unsorted_wavs_on_disk - unsorted_filenames)},
    {'metric': 'unique filenames in merged json',        'value': len(mr_filenames)},
    {'metric': 'wavs on disk in ALL/',                   'value': len(mr_wav_set)},
    {'metric': 'merged json files == ALL/ wavs',         'value': mr_filenames == mr_wav_set},
    {'metric': 'unsorted ∩ merged (filenames)',          'value': len(unsorted_filenames & mr_filenames)},
    {'metric': 'in unsorted but NOT in merged',          'value': len(unsorted_filenames - mr_filenames)},
    {'metric': 'in merged but NOT in unsorted',          'value': len(mr_filenames - unsorted_filenames)},
    {'metric': 'unsorted uids ∩ merged uids',            'value': len(unsorted_uids & mr_uids)},
])
overlap

**Reading.** The two sets *partially* overlap — neither is a strict superset:
- 1198 files are shared,
- 258 files exist only in `unsorted/`,
- 1811 files exist only in `medical_records/`.

So if we want everything, we'll need to merge both. If we only want the larger/newer snapshot, `medical_records/` covers ~83% of `unsorted/` files and adds 1.8k new ones. `medical_records/` is internally consistent: every JSON entry has a WAV on disk and vice versa. `unsorted/` typically has 3 json entries with no matching WAV (these come from `1604_final/`, where the JSON listed 391 files but only 388 wavs were actually shipped).

## 3. Audio file metadata

Sample 200 WAVs to confirm sample rate / channels / bit depth and estimate total duration. Header reads only — no decoding.

In [ ]:
random.seed(0)
sample = random.sample(mr_wav_files, 200)
sr_c, ch_c, sw_c, durs = Counter(), Counter(), Counter(), []
for name in sample:
    with contextlib.closing(wave.open(str(MR_WAVS / name), 'rb')) as w:
        sr_c[w.getframerate()]   += 1
        ch_c[w.getnchannels()]   += 1
        sw_c[w.getsampwidth()]   += 1
        durs.append(w.getnframes() / w.getframerate())

durs.sort()
p = lambda q: durs[int(q * (len(durs) - 1))]
audio_summary = pd.Series({
    'sample_rate':  dict(sr_c),
    'channels':     dict(ch_c),
    'sample_width': dict(sw_c),
    'duration_min_sec':    round(durs[0],  2),
    'duration_p25_sec':    round(p(0.25), 2),
    'duration_median_sec': round(p(0.50), 2),
    'duration_p75_sec':    round(p(0.75), 2),
    'duration_max_sec':    round(durs[-1], 2),
    'duration_mean_sec':   round(sum(durs) / len(durs), 2),
    'estimated_total_hours': round(sum(durs) / len(durs) * len(mr_wav_files) / 3600, 2),
})
audio_summary

In [ ]:
all_durs = []
dur_bad = []
for name in mr_wav_files:
    try:
        info = sf.info(str(MR_WAVS / name))
        all_durs.append(info.frames / info.samplerate)
    except Exception as e:
        dur_bad.append((name, str(e)))

all_durs = np.array(all_durs)
print(f'read {len(all_durs)} / {len(mr_wav_files)} files '
      f'(unreadable: {len(dur_bad)})')
print(f'total: {all_durs.sum() / 3600:.2f} h | '
      f'mean: {all_durs.mean():.2f} s | median: {np.median(all_durs):.2f} s | '
      f'min: {all_durs.min():.2f} s | max: {all_durs.max():.2f} s')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(all_durs, bins=60, color='#3b82f6', edgecolor='white')
axes[0].set_xlabel('duration (sec)')
axes[0].set_ylabel('files')
axes[0].set_title('Duration distribution — all medical_records/ wavs')
for q, label in [(0.5, 'median'), (0.95, 'p95'), (0.99, 'p99')]:
    v = np.quantile(all_durs, q)
    axes[0].axvline(v, color='#ef4444', linestyle='--', linewidth=1)
    axes[0].text(v, axes[0].get_ylim()[1] * 0.9, f' {label}={v:.1f}s',
                 color='#ef4444', fontsize=8)

axes[1].boxplot(all_durs, vert=False, showfliers=True)
axes[1].set_xlabel('duration (sec)')
axes[1].set_title('Boxplot (with outliers)')
fig.tight_layout()
plt.show()

pd.Series(all_durs).describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2)


Mostly 44.1 kHz mono int16 — the format the dnd inference pipeline (`pathology/pathology.py`) already handles (it resamples to 22050 Hz). A handful of files are 16384 Hz; pipelines should resample, not assume.

In [ ]:
import soundfile as sf

sr_full = Counter()
odd_sr_files = []
bad_files = []
for name in mr_wav_files:
    try:
        sr = sf.info(str(MR_WAVS / name)).samplerate
    except Exception as e:
        bad_files.append((name, str(e)))
        continue
    sr_full[sr] += 1
    if sr != 44100:
        odd_sr_files.append((name, sr))

print('sample-rate breakdown across all medical_records/ wavs:')
print(dict(sr_full))
print(f'non-44100: {len(odd_sr_files)} / {len(mr_wav_files)} '
      f'({100 * len(odd_sr_files) / len(mr_wav_files):.2f}%)')
print(f'unreadable: {len(bad_files)}')
odd_sr_files[:10]


## 4. Label distribution

Pathology vocabulary should match the 6 classes from `dnd/1/config.py`. Anything else is a labeling slip.

In [ ]:
def label_stats(files):
    interval_count = Counter()
    interval_secs  = defaultdict(float)
    file_level     = Counter()
    doctors        = Counter()

    for f in files:
        doctors[f.get('doctor', 'unknown')] += 1
        file_pathos = set()
        for m in f.get('markup', []) or []:
            p = m.get('pathology')
            if p is None: continue
            p = p.strip() if isinstance(p, str) else p
            if p in ('', 'Нет патологии (норма)'):
                p = 'none'
            interval_count[p] += 1
            interval_secs[p]  += float(m.get('length', 0) or 0)
            if p != 'none':
                file_pathos.add(p)
        if not file_pathos:
            file_level['none'] += 1
        elif len(file_pathos) == 1:
            file_level[next(iter(file_pathos))] += 1
        else:
            file_level['multi:' + '+'.join(sorted(file_pathos))] += 1
    return interval_count, interval_secs, file_level, doctors

uns_ic, uns_is, uns_fl, uns_dr = label_stats(all_unsorted_files)
mr_ic,  mr_is,  mr_fl,  mr_dr  = label_stats(mr_files)

In [ ]:
all_labels = sorted(set(uns_ic) | set(mr_ic))
label_table = pd.DataFrame({
    'unsorted_intervals':       [uns_ic.get(l, 0)         for l in all_labels],
    'unsorted_seconds':         [round(uns_is.get(l, 0), 1) for l in all_labels],
    'medical_records_intervals':[mr_ic.get(l, 0)          for l in all_labels],
    'medical_records_seconds':  [round(mr_is.get(l, 0), 1)  for l in all_labels],
    'in_canonical_6':           [l in PATHOLOGY_CLASSES    for l in all_labels],
}, index=all_labels).sort_values('medical_records_intervals', ascending=False)
label_table

**Things to act on:**
- `amphoric breathing`, `pleural friction`, `Жужжащие хрипы` (the Russian one is almost certainly meant to be `dry buzzing wheezing`) appear only in `unsorted/`, only a handful of times each. Not in the canonical 6. The dnd pipelines silently let these fall through.
- `weakened breathing` exists in both but is much more frequent in `medical_records/` (49 vs 2 intervals).
- `crepitus` is rare in both (~85–100 intervals). Stratification or class weighting is going to matter.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=False)
for ax, ic, title in [(axes[0], uns_ic, 'unsorted/ — interval counts'),
                      (axes[1], mr_ic,  'medical_records/ — interval counts')]:
    items = ic.most_common()
    labels, counts = zip(*items)
    colors = ['#3b82f6' if l in PATHOLOGY_CLASSES else '#ef4444' for l in labels]
    ax.barh(range(len(labels)), counts, color=colors)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xscale('log')
    ax.set_xlabel('intervals (log scale)')
fig.suptitle('Pathology label frequency — blue = canonical 6, red = out-of-vocab')
fig.tight_layout()
plt.show()

## 5. File-level pathology composition

What share of files have *zero* pathology, exactly one, or multiple co-occurring subtypes?

In [ ]:
def collapse(fl):
    none = fl.get('none', 0)
    single = sum(v for k, v in fl.items() if k != 'none' and not k.startswith('multi:'))
    multi  = sum(v for k, v in fl.items() if k.startswith('multi:'))
    total  = none + single + multi
    return {'none_only': none, 'single_pathology': single, 'multi_pathology': multi, 'total': total}

pd.DataFrame({'unsorted/': collapse(uns_fl), 'medical_records/': collapse(mr_fl)})

In [ ]:
top_combos = pd.Series(mr_fl).sort_values(ascending=False).head(15).rename('count').to_frame()
top_combos.index.name = 'file_level_pathology'
top_combos

Multi-pathology is real and not negligible (~17% of `medical_records/` files). The dnd `1/` pipeline handles this with `IGNORE_CONFLICTING_FRAMES = True` (frame target = -100, skipped by `CrossEntropyLoss`). Worth keeping in mind if we go in a different direction (e.g. multi-label sigmoid).

Let's listen some example

In [ ]:
wav, sr = torchaudio.load('../data/medical_records/ALL/000001213.wav')
print(wav.shape, sr)

In [ ]:
def visualize_audio(wav: torch.Tensor, sr: int = 22050, monoConvert: bool = False):
    """Plots the given waveform

    Args:
        wav (torch.Tensor): waveform (Nch, T) or (T,)
        sr (int, optional): sample rate. Defaults to 22050.
    """    
    if(monoConvert):
        # Average all channels
        if(len(wav.shape)>1):
            # Any to mono audio convertion
            wav = wav.mean(dim=0,keepdim=True)
        
        _,ax = plt.subplots(figsize=(14, 4))
        ax.plot(np.arange(0,wav.shape[1]/sr,1/sr),wav, alpha=.7, c='green')
        ax.grid()
        ax.set_xlabel('Time, sec', size=20)
        ax.set_ylabel('Amplitude', size=20)
    else:
        nChannels = wav.shape[0]
        if(nChannels==1):
            _,ax = plt.subplots(figsize=(14, 4))
            ax.plot(np.arange(0,wav.shape[1]/sr,1/sr),wav[0,:], alpha=.7, c='green')
            ax.grid()
            ax.set_xlabel('Time, sec', size=20)
            ax.set_ylabel('Amplitude', size=20)
        else:
            _, axs = plt.subplots(nChannels,1,figsize=(14, 4*nChannels))
            for id,ax in enumerate(axs):
                ax.plot(np.arange(0,wav.shape[1]/sr,1/sr),wav[id,:], alpha=.7, c='green')
                ax.grid()
                ax.set_xlabel('Time, sec', size=20)
                ax.set_ylabel(f'Amplitude, Ch={id+1}', size=20)
        
    plt.show()
    
    display.display(display.Audio(wav, rate=sr))

In [ ]:
visualize_audio(wav, sr)

In [ ]:

DATA_DIR = MR_DIR
WAV_DIR  = MR_WAVS
JSON_PATH = MR_JSON

In [ ]:
import matplotlib.patches as mpatches
from IPython import display

DATA_DIR = MR_DIR
WAV_DIR  = MR_WAVS
JSON_PATH = MR_JSON

N_FFT = 1024
HOP = 256
N_MELS = 128

# stable color per class
PATHOLOGY_COLORS = {
    'none':                   '#888888',
    'hard breathing':         '#1f77b4',
    'dry buzzing wheezing':   '#ff7f0e',
    'wet wheezing':           '#2ca02c',
    'dry whistling wheezing': '#d62728',
    'crepitus':               '#9467bd',
    'weakened breathing':     '#8c564b',
}

def to_mono(wav):
    return wav.mean(0, keepdim=True) if wav.shape[0] > 1 else wav

def compute_logspec_mel(wav, sr):
    spec = torchaudio.transforms.Spectrogram(n_fft=N_FFT, hop_length=HOP, power=2.0)
    mel  = torchaudio.transforms.MelSpectrogram(sample_rate=sr, n_fft=N_FFT,
                                                hop_length=HOP, n_mels=N_MELS, power=2.0)
    todb = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=80)
    return todb(spec(wav)), todb(mel(wav))

def _spec_imshow(ax, S, sr, title, kind='linear'):
    S = S.squeeze().numpy()
    duration = S.shape[-1] * HOP / sr
    ymax = sr / 2 if kind == 'linear' else S.shape[0]
    im = ax.imshow(S, origin='lower', aspect='auto',
                   extent=[0, duration, 0, ymax], cmap='magma')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Time, sec')
    ax.set_ylabel('Hz' if kind == 'linear' else 'mel bin')
    return im, duration, ymax

def overlay_pathologies(ax, markup, ymax=None, label_band=False):
    """Shade pathology spans on a time axis. If label_band, draw text band above."""
    if ymax is None:
        ymax = ax.get_ylim()[1]
    seen = set()
    for m in markup:
        p = m.get('pathology')
        if not p or p == 'none':
            continue
        if 'start' not in m or 'length' not in m:
            continue
        s, l = float(m['start']), float(m['length'])
        c = PATHOLOGY_COLORS.get(p, '#e377c2')
        ax.axvspan(s, s + l, color=c, alpha=0.25,
                   label=p if p not in seen else None)
        if label_band:
            ax.text(s + l / 2, ymax * 1.02, p, fontsize=7,
                    ha='center', va='bottom', color=c, rotation=0)
        seen.add(p)
    if seen:
        ax.legend(loc='upper right', fontsize=7, framealpha=0.8)


In [ ]:
def visualize_full(path):
    wav, sr = torchaudio.load(str(path))
    wav = to_mono(wav)
    logS, logM = compute_logspec_mel(wav, sr)

    # find markup for this file (if any)
    fname = path.name
    entry = next((f for f in meta if f['filename'] == fname), None)
    markup = entry['markup'] if entry else []

    fig, axes = plt.subplots(3, 1, figsize=(14, 11),
                             gridspec_kw={'height_ratios': [1, 1.2, 1.2]})

    # waveform
    t = np.arange(wav.shape[1]) / sr
    axes[0].plot(t, wav[0].numpy(), alpha=.7, c='green')
    axes[0].grid()
    axes[0].set_xlim(0, t[-1])
    axes[0].set_xlabel('Time, sec'); axes[0].set_ylabel('Amplitude')
    axes[0].set_title(f'{fname}  sr={sr}  shape={tuple(wav.shape)}')
    overlay_pathologies(axes[0], markup, ymax=axes[0].get_ylim()[1], label_band=True)

    # log spectrogram
    _, dur, ymax = _spec_imshow(axes[1], logS, sr, 'Log-spectrogram', kind='linear')
    overlay_pathologies(axes[1], markup, ymax=ymax)

    # log-mel
    _, dur, ymax = _spec_imshow(axes[2], logM, sr, 'Log-mel spectrogram', kind='mel')
    overlay_pathologies(axes[2], markup, ymax=ymax)

    plt.tight_layout()
    plt.show()
    display.display(display.Audio(wav.numpy(), rate=sr))

meta = json.load(open(JSON_PATH, encoding='utf-8-sig'))['files']
wav_files = sorted(WAV_DIR.glob('*.wav'))
visualize_full(random.choice(wav_files)) 
# visualize_full(WAV_DIR / "000126789.wav") # good example
# visualize_full(WAV_DIR / "000065694.wav") # good example


In [ ]:
# index segments by pathology class
by_class = {}
for f in meta:
    fname = f['filename']
    if not (WAV_DIR / fname).exists():
        continue
    for m in f.get('markup', []):
        p = m.get('pathology')
        if not p:
            continue
        if 'start' not in m or 'length' not in m:
            continue
        by_class.setdefault(p, []).append((fname, float(m['start']), float(m['length'])))

print({k: len(v) for k, v in by_class.items()})

def load_segment(fname, start, length):
    info = torchaudio.info(str(WAV_DIR / fname))
    sr = info.sample_rate
    frame_offset = max(0, int(start * sr))
    num_frames   = max(1, int(length * sr))
    wav, sr = torchaudio.load(str(WAV_DIR / fname),
                              frame_offset=frame_offset, num_frames=num_frames)
    return to_mono(wav), sr

N_PER_CLASS = 3

for cls in [c for c, segs in by_class.items() if len(segs) >= N_PER_CLASS]:
    picks = random.sample(by_class[cls], N_PER_CLASS)
    color = PATHOLOGY_COLORS.get(cls, '#e377c2')

    fig, axes = plt.subplots(3, N_PER_CLASS, figsize=(5 * N_PER_CLASS, 9))
    fig.suptitle(f'pathology: {cls}', fontsize=13, color=color, weight='bold')

    audios = []
    for j, (fname, start, length) in enumerate(picks):
        wav, sr = load_segment(fname, start, length)
        logS, logM = compute_logspec_mel(wav, sr)

        # waveform
        t = np.arange(wav.shape[1]) / sr
        axes[0, j].plot(t, wav[0].numpy(), alpha=.7, c=color)
        axes[0, j].grid()
        axes[0, j].set_xlim(0, max(t[-1], 1e-3))
        axes[0, j].set_xlabel('Time, sec'); axes[0, j].set_ylabel('Amplitude')
        axes[0, j].set_title(f'{fname}\nt=[{start:.2f}, {start+length:.2f}]s', fontsize=9)
        # outline border in class color so it's visually grouped
        for spine in axes[0, j].spines.values():
            spine.set_edgecolor(color); spine.set_linewidth(1.5)

        _spec_imshow(axes[1, j], logS, sr, 'log-spec', kind='linear')
        _spec_imshow(axes[2, j], logM, sr, 'log-mel',  kind='mel')

        audios.append((fname, wav, sr))

    plt.tight_layout()
    plt.show()
    for fname, wav, sr in audios:
        print(f'  {fname}  sr={sr}  dur={wav.shape[1]/sr:.2f}s')
        display.display(display.Audio(wav.numpy(), rate=sr))


## 6. Doctor / annotator distribution

Same files sometimes appear under multiple doctor IDs — this is where label disagreement hides.

In [ ]:
all_doctors = sorted(set(uns_dr) | set(mr_dr))
doctors_df = pd.DataFrame({
    'unsorted_entries':        [uns_dr.get(d, 0) for d in all_doctors],
    'medical_records_entries': [mr_dr.get(d, 0)  for d in all_doctors],
}, index=[f'doctor_{d}' for d in all_doctors]).sort_values('medical_records_entries', ascending=False)
doctors_df

In [ ]:
# Files with multiple JSON entries (likely re-labeled by another doctor).
from collections import Counter as C
mr_dups  = [fn for fn, c in C(e['filename'] for e in mr_files).items() if c > 1]
uns_dups = [fn for fn, c in C(e['filename'] for e in all_unsorted_files).items() if c > 1]
print(f'merged json:    {len(mr_dups)} filenames appear multiple times')
print(f'unsorted jsons: {len(uns_dups)} filenames appear multiple times')

# Spot-check: same filename, different doctors?
by_filename = defaultdict(list)
for e in mr_files:
    by_filename[e['filename']].append(e['doctor'])
diff_doctor = {fn: ds for fn, ds in by_filename.items() if len(set(ds)) > 1}
print(f'merged json: {len(diff_doctor)} filenames labeled by >1 distinct doctor')

## 7. Takeaways for the next EDA step

1. **`medical_records/` is the larger, newer, internally-consistent snapshot** — wavs on disk match JSON 1:1. Use it as the primary source.
2. **`unsorted/` adds 258 files that aren't in `medical_records/`**. If we want max coverage, merge both, dedup by `uid`, and resolve ~100 cases of multiple doctor labels.
3. **Out-of-vocab labels are tiny** (`amphoric breathing`, `pleural friction`, the stray Russian one) — a few intervals total. Drop or remap.
4. **Class imbalance is severe**: `none` ≈ 5170 intervals vs `crepitus` ≈ 86. Whatever model we end up training will need class weighting, focal loss, or oversampling.
5. **Audio is uniform enough** (44.1 kHz mono int16, ~9 s median) that no preprocessing surprises are expected. Resample to whatever the chosen feature extractor wants (16 kHz for wav2vec/HuBERT, 22050 Hz for the Tortoise mini-encoder).
6. **Multi-pathology files are ~17% of the corpus** — pick a target formulation (single-label with conflict frames ignored, vs multi-label sigmoid) before further work.
7. **`unsorted/` is now in extracted form**, with `data/archives/all.tar` as the cold backup. The dnd pipelines (`1/dataset.py`, `script/training/code/core_pipeline.py`) read from zip archives, so if we want to plug this data into them as-is we'll need either (a) a small loader change to walk subdirectories, or (b) re-zipping each batch.